In [2]:
!pip install -U bitsandbytes>=0.46.1

In [22]:
!pip install argostranslate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 52.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 80.2 MB/s eta 0:00:00:00:0100:01


In [3]:
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig

# This config is the secret to running huge models for free
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = LlavaForConditionalGeneration.from_pretrained(
    "llava-hf/llava-1.5-7b-hf", 
    quantization_config=quant_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

In [17]:
from transformers import AutoProcessor

model_id = "llava-hf/llava-1.5-7b-hf"
processor = AutoProcessor.from_pretrained(model_id)

print("Processor is now defined and ready!")

processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/41.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

Processor is now defined and ready!


In [24]:
import argostranslate.package
import argostranslate.translate

# Download and install Hindi -> English and English -> Hindi packages
argostranslate.package.update_package_index()
available_packages = argostranslate.package.get_available_packages()
package_to_install = next(
    filter(
        lambda x: x.from_code == "en" and x.to_code == "hi", available_packages
    )
)
argostranslate.package.install_from_path(package_to_install.download())

In [25]:
from PIL import Image
import torch

# 1. Use the path you copied from the sidebar
# Example: '/kaggle/input/vizwiz/train/VizWiz_train_00000000.jpg' 
image_path = "/kaggle/working/images/cube.png"

try:
    image = Image.open(image_path).convert("RGB")
    print("Image loaded successfully!")
except Exception as e:
    print(f"Error: {e}. Check if the path is correct!")

# 2. Your Hindi Question 
prompt = "USER: <image>\nइस चित्र में क्या है? ASSISTANT:" 

# 3. Process and Generate
# We use .to(device="cuda", dtype=torch.float16) to match the GPU settings
inputs = processor(text=prompt, images=image, return_tensors="pt").to("cuda", torch.float16)

# Better generation settings for Hindi
output = model.generate(
    **inputs, 
    max_new_tokens=50,
    do_sample=True,      # Allows the model to be a bit more creative/flexible
    temperature=0.7,     # Prevents the model from getting stuck in a loop
    top_p=0.9,           # Only considers the most likely words
    repetition_penalty=1.2 # Forces the model NOT to repeat the same Hindi words
)
english_response = processor.decode(output[0], skip_special_tokens=True).split("ASSISTANT:")[-1].strip()

hindi_translation = argostranslate.translate.translate(english_response, "en", "hi")

print("--- VQA Results ---")
print(f"English (Original): {english_response}")
print(f"Hindi (Translated): {hindi_translation}")

Image loaded successfully!


2026-03-10 04:26:47 WARNING: Language en package default expects mwt, which has been added


--- VQA Results ---
English (Original): This image features a colorful, three-dimensional Rubik's Cube. The cube is made up of various colored squares that are arranged in different patterns across its surface. These colors and patterns make the Rubik's Cube visually
Hindi (Translated): इस छवि में एक रंगीन, त्रि-आयामी रूबिक क्यूब है। घन विभिन्न रंगीन वर्गों से बना है जो इसकी सतह के विभिन्न पैटर्नों में व्यवस्थित होते हैं। ये रंग और पैटर्न रूबिक के क्यूब को नेत्रहीन रूप से बनाते हैं
